In [ ]:
!pip install -q qwen-vl-utils bitsandbytes accelerate

In [ ]:
import pandas as pd
import torch
import gc
import numpy as np
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from qwen_vl_utils import process_vision_info

# ==========================================
# 1. KAGGLE PATH SETUP
# ==========================================
PROJECT_DIR = Path("/kaggle/input/competitions/museumscat-specimen-collection-annotation-task/")
TEST_CSV_PATH = PROJECT_DIR / "test.csv"
IMAGES_DIR = PROJECT_DIR / "images"

test_df = pd.read_csv(TEST_CSV_PATH)
print(f"Total test images to process: {len(test_df)}")

# ==========================================
# 2. OPTIMIZED MODEL LOADING (4-BIT)
# ==========================================
device = "cuda" if torch.cuda.is_available() else "cpu"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

print("Loading Qwen2.5-VL onto Kaggle GPU...")
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2.5-VL-3B-Instruct",
    quantization_config=bnb_config,
    device_map="auto"
)
processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-3B-Instruct", padding_side="left")

# Strict output prompt structure
PROMPT = """Extract the collection Date and Locality from this museum label. 
Output STRICTLY in this exact format:
Date: <date> | Locality: <locality>
If a value is unreadable or missing, use MISSING. Do not add any other text."""

def geo_mean(probs):
    if not len(probs):
        return 0.0
    arr = np.clip(probs, 1e-9, 1.0)
    return float(np.exp(np.mean(np.log(arr))))

In [ ]:
results = []

for idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Extracting True Logits"):
    img_filename = row['image_file']
    img_path = IMAGES_DIR / img_filename
    
    p_date, p_loc = "MISSING", "MISSING"
    c_date, c_loc = 0.0, 0.0
    
    if img_path.exists():
        img = Image.open(img_path).convert("RGB")
        messages = [{
            "role": "user",
            "content": [
                {"type": "image", "image": img, "max_pixels": 262144}, 
                {"type": "text", "text": PROMPT},
            ],
        }]
        
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        image_inputs, video_inputs = process_vision_info(messages)
        
        inputs = processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt",
        ).to(device)
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs, 
                max_new_tokens=64, 
                return_dict_in_generate=True, 
                output_scores=True
            )
            
        input_length = inputs.input_ids.shape[1]
        generated_tokens = outputs.sequences[:, input_length:]
        
        transition_scores = model.compute_transition_scores(
            outputs.sequences, outputs.scores, normalize_logits=True
        )
        token_probs = torch.exp(transition_scores)[0].cpu().numpy()
        token_ids = generated_tokens[0].cpu().numpy()
        
        date_probs = []
        loc_probs = []
        decoded_text = ""
        
        for t_id, prob in zip(token_ids, token_probs):
            token_str = processor.decode(t_id)
            decoded_text += token_str
            
            if "Date:" in decoded_text and "|" not in decoded_text:
                if "Date:" not in token_str: 
                    date_probs.append(prob)
            elif "Locality:" in decoded_text:
                if "Locality:" not in token_str:
                    loc_probs.append(prob)

        if " | Locality: " in decoded_text:
            try:
                parts = decoded_text.split(" | Locality: ")
                p_date = parts[0].replace("Date: ", "").strip()
                p_loc = parts[1].strip()
            except Exception:
                pass
                
        c_date = geo_mean(date_probs) if p_date != "MISSING" and date_probs else 0.0
        c_loc = geo_mean(loc_probs) if p_loc != "MISSING" and loc_probs else 0.0
        
        del inputs, outputs, generated_tokens, transition_scores
        torch.cuda.empty_cache()
        gc.collect()

    results.append({
        "image_file": img_filename,
        "verbatimDate": p_date,
        "verbatimDate_confidence": c_date,
        "verbatimLocality": p_loc,
        "verbatimLocality_confidence": c_loc
    })

# Save your final leaderboard submission
submission_df = pd.DataFrame(results)
submission_df.to_csv("submission.csv", index=False)
print("\nSuccess! submission.csv has been written to your output directory.")